# G7 — Baseline U-Net AP (Reference Upper Bound)

**Goal:** Train baseline U-Net, then compute AP using its softmax outputs as proxy heatmaps.

**Why:** Provides the reference point for the Two-Barrier Framework:
- U-Net AP (using output softmax) = segmentation precision ≈ upper bound
- ProtoSegNet AP = prototype spatial alignment quality
- Large gap → prototypes fail to spatially locate structures (bypass barrier)
- Small gap → prototypes successfully locate structures

The checkpoint `checkpoints/baseline_unet_ct.pth` is missing — retrain from scratch.

In [ ]:
import sys, os, time
_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
os.chdir(_root)
sys.path.insert(0, _root)
os.environ.setdefault('PYTORCH_MPS_HIGH_WATERMARK_RATIO', '0.0')
os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')

import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

from src.data.mmwhs_dataset import MMWHSSliceDataset, make_dataloaders, NUM_CLASSES, LABEL_NAMES
from src.models.unet import UNet2D
from src.losses.segmentation import SegmentationLoss
from src.metrics.dice import dice_per_class, mean_foreground_dice

DEVICE   = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
DATA_DIR = 'data/pack/processed_data'
CKPT_DIR = Path('checkpoints')
OUT_DIR  = Path('results/v10')
OUT_DIR.mkdir(parents=True, exist_ok=True)
FOREGROUND = list(range(1, NUM_CLASSES))
CLASS_NAMES = [LABEL_NAMES[k] for k in range(NUM_CLASSES)]

print(f'Device: {DEVICE}')

## 1. Train baseline U-Net (~20 epochs)

In [ ]:
CKPT_PATH = CKPT_DIR / 'baseline_unet_ct.pth'

loaders = make_dataloaders(DATA_DIR, 'ct', batch_size=16)
class_weights = torch.load('data/class_weights_ct.pt', weights_only=True)
print(f'Train: {len(loaders["train"].dataset)}  Val: {len(loaders["val"].dataset)}')

model = UNet2D(n_classes=NUM_CLASSES).to(DEVICE)
seg_loss = SegmentationLoss(class_weights=class_weights.to(DEVICE), n_classes=NUM_CLASSES)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=25)

best_val = 0.0
patience_count = 0
PATIENCE = 5
MAX_EPOCHS = 25

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    t0 = time.time()
    total_loss, n_batches = 0.0, 0
    for batch in loaders['train']:
        imgs = batch['image'].to(DEVICE)
        lbls = batch['label'].to(DEVICE)
        optimizer.zero_grad()
        logits = model(imgs)
        loss = seg_loss(logits, lbls)['loss']
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        n_batches += 1
    scheduler.step()

    # Validate
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for batch in loaders['val']:
            logits = model(batch['image'].to(DEVICE))
            all_logits.append(logits.cpu())
            all_labels.append(batch['label'])
    val_dice = mean_foreground_dice(dice_per_class(torch.cat(all_logits), torch.cat(all_labels)))

    improved = val_dice > best_val
    if improved:
        best_val = val_dice
        patience_count = 0
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                    'best_val_dice': best_val}, CKPT_PATH)
    else:
        patience_count += 1

    flag = '✓' if improved else ' '
    print(f'  [{flag}] Ep {epoch:3d} | loss={total_loss/n_batches:.4f} | val={val_dice:.4f} | {time.time()-t0:.0f}s', flush=True)
    if patience_count >= PATIENCE:
        print(f'Early stopping at epoch {epoch}')
        break

print(f'\nBest val Dice: {best_val:.4f}  →  {CKPT_PATH}')

## 2. Load best checkpoint

In [ ]:
ckpt = torch.load(CKPT_PATH, map_location='cpu', weights_only=False)
model = UNet2D(n_classes=NUM_CLASSES).to(DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'Loaded: epoch={ckpt["epoch"]}  best_val={ckpt["best_val_dice"]:.4f}')

## 3. Compute AP using softmax output as proxy heatmap

For each class k, treat softmax[:, k, :, :] as the activation map.
Threshold at 95th percentile → top-5% mask → precision vs GT.
This gives the AP a model could achieve if its intermediate representations
perfectly matched the segmentation output — the practical upper bound.

In [ ]:
test_ds  = MMWHSSliceDataset(DATA_DIR, 'ct', 'test',  augment=False, preload=True)
test_loader = torch.utils.data.DataLoader(test_ds, batch_size=16, shuffle=False)

PERCENTILE = 95.0
acc = defaultdict(list)  # class_idx → list of per-slice AP values

with torch.no_grad():
    for batch in test_loader:
        images = batch['image'].to(DEVICE)
        labels = batch['label'].to(DEVICE)
        logits = model(images)            # (B, K, H, W)
        probs  = logits.softmax(dim=1)    # (B, K, H, W) — proxy heatmap
        B, H, W = labels.shape

        for k in FOREGROUND:
            A_k = probs[:, k, :, :]        # (B, H, W)
            G_k = (labels == k).float()    # (B, H, W)
            for b in range(B):
                a_flat = A_k[b].flatten()
                thresh = torch.quantile(a_flat, PERCENTILE / 100.0)
                M_k = (A_k[b] >= thresh).float().flatten()
                if M_k.sum() < 1:
                    continue
                ap = ((M_k * G_k[b].flatten()).sum() / M_k.sum()).item()
                acc[k].append(ap)

rows = []
for k in FOREGROUND:
    ap_val = float(np.mean(acc[k])) if acc[k] else float('nan')
    rows.append({'class_idx': k, 'class_name': CLASS_NAMES[k], 'ap': ap_val})

ap_df = pd.DataFrame(rows)
mean_ap = ap_df['ap'].mean()

ap_df.to_csv(OUT_DIR / 'xai_ap_baseline_unet.csv', index=False)

print('Baseline U-Net AP (proxy heatmap = softmax output):')
print(ap_df.round(3).to_string(index=False))
print(f'\nMean AP (foreground): {mean_ap:.4f}')

## 4. Summary and comparison

In [ ]:
summary = pd.DataFrame([
    {'model': 'Baseline U-Net',     'dice': ckpt['best_val_dice'], 'ap': mean_ap,  'note': 'proxy heatmap = softmax output'},
    {'model': 'Stage 29 (skip L3+L4)', 'dice': 0.821,             'ap': 0.051,    'note': 'prototype heatmaps, bypass active'},
    {'model': 'Stage 8A (skip L4)',  'dice': 0.810,               'ap': 0.057,    'note': 'prototype heatmaps, bypass active'},
    {'model': '9b (no-skip L3+L4)', 'dice': 0.559,               'ap': 0.301,    'note': 'prototype heatmaps, no bypass'},
    {'model': '9a (no-skip L4)',     'dice': 0.606,               'ap': 0.312,    'note': 'prototype heatmaps, no bypass'},
])
summary.to_csv(OUT_DIR / 'xai_summary_g7_comparison.csv', index=False)

print('AP Comparison — U-Net vs Prototype Networks')
print('='*70)
print(summary[['model','dice','ap','note']].round(3).to_string(index=False))
print()
print('Key insight:')
print(f'  U-Net AP (output logits)  = {mean_ap:.3f}  ← upper bound: segmentation IS the heatmap')
print(f'  Skip prototypes AP        ~ 0.051–0.057  ← bypass: decoder ignores prototypes')
print(f'  No-skip prototypes AP     ~ 0.301–0.312  ← bypass removed: 5-6× closer to U-Net')
print()
print(f'Gap to U-Net: skip={mean_ap-0.054:.3f}, no-skip={mean_ap-0.307:.3f}')